# 04 — Compute table: parameters, FLOPs/MACs, latency, training time (all 12 configurations)

**Paper sections fed:** new compute table (IV), V-C deployment claim. Runs on CPU; GPU latency is added if a GPU is
present (≈5 min — no training).

* **Parameters:** total, trainable (as fine-tuned in the paper), attention-module only.
* **FLOPs:** `torch.utils.flop_counter.FlopCounterMode` for one 3×224×224 image; counts conv / linear / matmul
  (the attention gates' element-wise multiplies, BN, activations and pooling are not counted — <1% here).
  MACs = FLOPs / 2.
* **Latency:** median of timed forward passes, batch 1 (and 32 on GPU), fp32, after warm-up.
* **Training time:** read from notebook 02's `metrics.json` (mean ± SD over seeds). These depend on Kaggle hardware
  and AMP setting; report them as indicative.

Note for the paper: MobileNetV2 with a 4-class head has **2.23M** parameters (3.4–3.5M is the 1000-class ImageNet
model) and ResNet50 **23.5M** (25.6M with the ImageNet head) — the counts below are the ones to cite.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# SHARED CODE — identical copy in notebooks 02, 04 and 05.
# Re-implements the 24 original notebooks in one place. Architectures, freezing,
# learning rates, loss, augmentation and schedule match the originals exactly;
# deviations are explicit, named options (see ORIGINAL-PIPELINE NOTES below).
# ═════════════════════════════════════════════════════════════════════════════
import os, re, json, time, math, random, hashlib, glob, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

CLASS_NAMES = ['COVID19', 'NORMAL', 'PNEUMONIA', 'TURBERCULOSIS']  # folder names (dataset spells TB this way)
NUM_CLASSES = 4
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

# ── ORIGINAL-PIPELINE NOTES (verified by reading the 20 shared notebooks) ──────
# * ResNet50: layer4 fine-tuned @1e-4; attention + fc @1e-3.
# * DenseNet121: denseblock4 + norm5 @1e-4; ReLU -> [attn] -> GAP -> fc @1e-3.
# * MobileNetV2: features[18] (final 1x1 conv 320->1280) @1e-4; [attn] + fc @1e-3.
#   The ORIGINAL BASELINE kept torchvision's Dropout(0.2) before fc, while the
#   SE/CBAM variants used no dropout. Replicated by default (head_dropout=None);
#   pass opts={'head_dropout': 0.0} to remove the confound.
# * Custom CNN: all layers @1e-3 from scratch; head 512->256->ReLU->Dropout(0.5)->4.
# * "Frozen" layers have requires_grad=False but stay in train() mode, so their
#   BatchNorm running statistics still update during training (replicated).
# * SE/CBAM hidden width = max(C//16, 4). Linear vs 1x1-conv MLP forms used in
#   different originals are mathematically equivalent.

# ─────────────────────────────── attention blocks ─────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
                                nn.Linear(hidden, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c, _, _ = x.size()
        return x * self.fc(self.avg_pool(x).view(b, c)).view(b, c, 1, 1)

class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.shared_mlp = nn.Sequential(nn.Linear(channels, hidden, bias=False), nn.ReLU(inplace=True),
                                        nn.Linear(hidden, channels, bias=False))
        self.avg_pool, self.max_pool = nn.AdaptiveAvgPool2d(1), nn.AdaptiveMaxPool2d(1)
    def forward(self, x):
        b, c, _, _ = x.size()
        s = self.shared_mlp(self.avg_pool(x).view(b, c)) + self.shared_mlp(self.max_pool(x).view(b, c))
        return x * torch.sigmoid(s).view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
    def forward(self, x):
        m = torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], 1)
        return x * torch.sigmoid(self.conv(m))

class CBAM(nn.Module):
    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(channels, reduction)
        self.spatial_attention = SpatialAttention(kernel_size)
    def forward(self, x):
        return self.spatial_attention(self.channel_attention(x))

def make_attention(kind, channels):
    if kind == 'none': return nn.Identity()
    if kind == 'se':   return SEBlock(channels)
    if kind == 'cbam': return CBAM(channels)
    raise ValueError(kind)

# ─────────────────────────────── unified network ──────────────────────────────
class XRVInput(nn.Module):
    """For torchxrayvision (CXR-pretrained) weights: undo the ImageNet normalisation of channel 0
    and map [0, 1] intensities to the [-1024, 1024] single-channel range those weights expect.
    Keeps the data pipeline identical for every model."""
    def forward(self, x):
        return (x[:, :1] * STD[0] + MEAN[0]) * 2048.0 - 1024.0

class ReLUWrap(nn.Module):           # DenseNet's functional ReLU after features
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x): return F.relu(self.m(x))

def custom_cnn_features():
    layers, cin = [], 3
    for cout in [32, 64, 128, 256, 512]:
        layers += [nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2)]
        cin = cout
    return nn.Sequential(*layers)

class CXRNet(nn.Module):
    """body -> attn -> GAP -> head.  `attn` output (== input to GAP) is the
    uniform Grad-CAM target for all 12 configurations."""
    def __init__(self, arch, attention='none', pretrained=True, unfreeze='last', head_dropout=None,
                 pretrain='imagenet', placement='single'):
        super().__init__()
        self.arch, self.attention, self.placement = arch, attention, placement
        extra = []                                   # attention modules besides the pre-pool one
        if placement not in ('single', 'multi'): raise ValueError(placement)
        if placement == 'multi' and (arch != 'resnet50' or attention == 'none'):
            raise ValueError('multi-depth placement is implemented for ResNet50 with SE/CBAM only')
        if arch == 'resnet50':
            b = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
            l3, l4 = b.layer3, b.layer4
            if placement == 'multi':                 # canonical SE-/CBAM-ResNet position: inside the residual branch
                for blk in list(b.layer3) + list(b.layer4):          # of every bottleneck of layer3 (6) and layer4 (3),
                    m = make_attention(attention, blk.bn3.num_features)  # after bn3, before the identity addition,
                    blk.bn3 = nn.Sequential(blk.bn3, m); extra.append(m)  # so the pretrained identity path is untouched;
                                                                         # + the pre-pool self.attn = 10 modules
            self.body = nn.Sequential(nn.Sequential(b.conv1, b.bn1, b.relu, b.maxpool), b.layer1, b.layer2, l3, l4)
            C = 2048
            ft = {'last': [self.body[4]], 'last2': [self.body[3], self.body[4]]}[unfreeze]
        elif arch == 'densenet121':
            if pretrain == 'xrv':                    # CXR-domain pretraining (torchxrayvision, 1-channel input)
                try:
                    import torchxrayvision as xrv
                except ImportError:                  # not preinstalled on Kaggle (needs Internet on)
                    import subprocess, sys
                    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torchxrayvision'])
                    import torchxrayvision as xrv
                b = xrv.models.DenseNet(weights='densenet121-res224-all' if pretrained else None)
                f = b.features; self.body = nn.Sequential(XRVInput(), ReLUWrap(f))
            elif pretrain == 'imagenet':
                b = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
                f = b.features; self.body = ReLUWrap(f)
            else:
                raise ValueError(pretrain)
            C = 1024
            ft = {'last': [f.denseblock4, f.norm5],
                  'last2': [f.denseblock3, f.transition3, f.denseblock4, f.norm5]}[unfreeze]
        elif arch == 'mobilenetv2':
            if pretrain != 'imagenet': raise ValueError('CXR pretraining implemented for DenseNet121 only')
            b = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None)
            self.body = b.features; C = 1280
            ft = {'last': [b.features[18]], 'last2': [b.features[17], b.features[18]]}[unfreeze]
        elif arch == 'customcnn':
            self.body = custom_cnn_features(); C = 512; ft = None
        else:
            raise ValueError(arch)
        self.attn = make_attention(attention, C)
        self.pool = nn.AdaptiveAvgPool2d(1)
        if arch == 'customcnn':
            self.head = nn.Sequential(nn.Linear(C, 256), nn.ReLU(inplace=True), nn.Dropout(0.5), nn.Linear(256, NUM_CLASSES))
        else:
            if head_dropout is None:   # replicate originals
                head_dropout = 0.2 if (arch == 'mobilenetv2' and attention == 'none') else 0.0
            self.head = nn.Sequential(nn.Dropout(head_dropout), nn.Linear(C, NUM_CLASSES)) if head_dropout > 0 \
                        else nn.Sequential(nn.Linear(C, NUM_CLASSES))
        self.channels = C
        # freezing
        if ft is not None:
            for p in self.body.parameters(): p.requires_grad = False
            for m in ft:
                for p in m.parameters(): p.requires_grad = True
        self._extra_attn = extra                     # plain list: not registered twice
        for m in extra:
            for p in m.parameters(): p.requires_grad = True
        self._ft_modules = ft

    def attention_parameters(self):
        ps = list(self.attn.parameters())
        for m in self._extra_attn: ps += list(m.parameters())
        return ps

    def forward(self, x):
        x = self.attn(self.body(x))
        return self.head(torch.flatten(self.pool(x), 1))

    def param_groups(self, lr_ft=1e-4, lr_new=1e-3, lr_attn=None, lr_scratch=1e-3):
        if self.arch == 'customcnn':
            return [{'params': list(self.parameters()), 'lr': lr_scratch, 'name': 'all'}]
        lr_attn = lr_new if lr_attn is None else lr_attn
        att = self.attention_parameters(); att_ids = {id(p) for p in att}
        g = [{'params': [p for m in self._ft_modules for p in m.parameters() if id(p) not in att_ids],
              'lr': lr_ft, 'name': 'finetuned_block'}]
        if self.attention != 'none':
            g.append({'params': att, 'lr': lr_attn, 'name': 'attention'})
        g.append({'params': list(self.head.parameters()), 'lr': lr_new, 'name': 'head'})
        return g

def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    att = sum(p.numel() for p in model.attention_parameters())
    return {'params_total': tot, 'params_trainable': tr, 'params_attention': att}

# ─────────────────────────────── data ─────────────────────────────────────────
def list_split(root, split):
    items = []
    for ci, c in enumerate(CLASS_NAMES):
        d = os.path.join(root, split, c)
        if not os.path.isdir(d): raise FileNotFoundError(d)
        for fn in sorted(os.listdir(d)):               # same order as ImageFolder
            if fn.lower().endswith(IMG_EXTS):
                items.append((os.path.join(d, fn), ci))
    return items

def load_256(path, box=None):
    """Grayscale(3) -> Resize((256,256)) of the originals, cached as 1 channel.
    Converting back with .convert('RGB') reproduces the original tensors exactly
    (verified in the smoke test). `box` = (x0, y0, x1, y1) in [0, 1] crops the image first
    (lung-crop preprocessing; boxes from notebook 07)."""
    img = Image.open(path).convert('RGB').convert('L')   # == ImageFolder loader + Grayscale
    if box is not None:
        W, H = img.size; x0, y0, x1, y1 = box
        img = img.crop((int(round(x0 * W)), int(round(y0 * H)), int(round(x1 * W)), int(round(y1 * H))))
    img = img.resize((256, 256), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)

_CACHE = {}
def cached_arrays(items, tag, verbose=True, boxes=None):
    key = (tag, len(items), items[0][0] if items else '')
    if key not in _CACHE:
        t = time.time()
        _CACHE[key] = np.stack([load_256(p, None if boxes is None else boxes.get(p)) for p, _ in items])
        if verbose: print(f'  cached {tag}: {len(items)} images in {time.time()-t:.0f}s')
    return _CACHE[key]

train_tf = transforms.Compose([
    transforms.RandomCrop(224),
    transforms.RandomAffine(degrees=5, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
# Paper Sec. III-B oversampling augmentation (applied once per synthetic copy, fixed per copy)
oversample_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=7, translate=(0.04, 0.04), scale=(0.96, 1.04)),
    transforms.ColorJitter(brightness=0.08, contrast=0.08)])

class CXRDataset(Dataset):
    """index entries: (array_row, label, copy_id). copy_id>0 => synthetic copy that
    receives a FIXED oversampling augmentation (like an offline-generated file)."""
    def __init__(self, arrays, labels, index, tf, seed=0):
        self.a, self.y, self.index, self.tf, self.seed = arrays, labels, index, tf, seed
    def __len__(self): return len(self.index)
    def __getitem__(self, i):
        r, y, copy_id = self.index[i]
        img = Image.fromarray(self.a[r]).convert('RGB')
        if copy_id > 0:
            with torch.random.fork_rng(devices=[]):
                torch.manual_seed((self.seed * 1_000_003 + r * 1009 + copy_id) % (2**31))
                img = oversample_tf(img)
        return self.tf(img), y

def oversample_index(labels, seed):
    """Keep every original image; add fixed-augmented copies of minority classes
    until each class matches the majority count (no downsampling)."""
    labels = np.asarray(labels); rng = np.random.RandomState(seed)
    counts = np.bincount(labels, minlength=NUM_CLASSES); target = counts.max()
    idx = [(r, int(labels[r]), 0) for r in range(len(labels))]
    for c in range(NUM_CLASSES):
        rows = np.where(labels == c)[0]; need = target - len(rows); copy = 1
        while need > 0:
            order = rng.permutation(rows)
            for r in order[:need]:
                idx.append((int(r), c, copy))
            need -= min(need, len(order)); copy += 1
    return idx

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def _worker_init(wid):
    s = torch.initial_seed() % 2**32; np.random.seed(s); random.seed(s)

def patient_key(path):
    """Patient identifier recoverable from the filename (Kermany-style Pneumonia/Normal names);
    COVID-19 and Tuberculosis filenames carry none, so each image is its own group."""
    fn = os.path.basename(path)
    m = re.match(r'person(\d+)_', fn)
    if m: return 'P' + m.group(1)
    m = re.match(r'(NORMAL2-)?IM-(\d+)-', fn)
    if m: return ('N2-' if m.group(1) else 'N-') + m.group(2)
    return 'F:' + fn

def grouped_val_split(items, arrays, frac=0.15, seed=12345):
    """Hold out ~frac of each class as validation, keeping patients and pixel-identical duplicates together.
    The split is fixed (independent of the training seed). Returns (train_rows, val_rows)."""
    parent = {}
    def find(k):
        while parent.setdefault(k, k) != k: parent[k] = parent[parent[k]]; k = parent[k]
        return k
    keys = [patient_key(p) for p, _ in items]; first_with_hash = {}
    for r, k in enumerate(keys):
        h = hashlib.md5(arrays[r].tobytes()).hexdigest()
        if h in first_with_hash: parent[find(k)] = find(first_with_hash[h])
        else: first_with_hash[h] = k
    groups = {}
    for r, k in enumerate(keys): groups.setdefault(find(k), []).append(r)
    labels = np.array([y for _, y in items]); rng = np.random.RandomState(seed); val = set()
    for c in range(NUM_CLASSES):
        gs = sorted({find(keys[r]) for r in np.where(labels == c)[0]}); rng.shuffle(gs)
        target, n = frac * (labels == c).sum(), 0
        for g in gs:
            if n >= target: break
            rows = [r for r in groups[g] if r not in val]; val.update(groups[g]); n += sum(labels[r] == c for r in rows)
    val_rows = sorted(val); tr_rows = [r for r in range(len(items)) if r not in val]
    return tr_rows, val_rows

# ─────────────────────────────── metrics ──────────────────────────────────────
def confusion(y, p, k=NUM_CLASSES):
    return np.bincount(np.asarray(y) * k + np.asarray(p), minlength=k * k).reshape(k, k)

def metrics_from_cm(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; tn = cm.sum() - tp - fp - fn
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    spec = np.divide(tn, tn + fp, out=np.zeros_like(tp), where=(tn + fp) > 0)
    out = {'acc': tp.sum() / cm.sum(), 'macro_f1': f1.mean(), 'balanced_acc': rec.mean(),
           'weighted_f1': (f1 * cm.sum(1)).sum() / cm.sum()}
    for i, c in enumerate(CLASS_NAMES):
        out[f'prec_{c}'], out[f'rec_{c}'], out[f'f1_{c}'], out[f'spec_{c}'] = prec[i], rec[i], f1[i], spec[i]
    return out

def macro_f1_fast(y, p, k=NUM_CLASSES):
    return metrics_from_cm(confusion(y, p, k))['macro_f1']

In [ ]:
import time, platform
from torch.utils.flop_counter import FlopCounterMode
RESULT_GLOBS = ['/kaggle/input/**/results/*/metrics.json', '/kaggle/working/results/*/metrics.json']
OUT = '/kaggle/working/compute'; os.makedirs(OUT, exist_ok=True)
N_TIMED, N_WARM = 50, 10
torch.set_num_threads(max(1, os.cpu_count() or 1))
gpu = torch.device('cuda') if torch.cuda.is_available() else None
ARCH_LABEL = {'resnet50': 'ResNet50', 'densenet121': 'DenseNet121', 'mobilenetv2': 'MobileNetV2', 'customcnn': 'Custom CNN'}
ATT_LABEL = {'none': 'Baseline', 'se': 'SE', 'cbam': 'CBAM'}

def flops(model):
    model.eval(); x = torch.randn(1, 3, 224, 224)
    with FlopCounterMode(display=False) as fc, torch.no_grad(): model(x)
    return fc.get_total_flops()

@torch.no_grad()
def latency_ms(model, dev, bs):
    model = model.to(dev).eval(); x = torch.randn(bs, 3, 224, 224, device=dev)
    for _ in range(N_WARM): model(x)
    ts = []
    for _ in range(N_TIMED):
        if dev.type == 'cuda': torch.cuda.synchronize()
        t = time.perf_counter(); model(x)
        if dev.type == 'cuda': torch.cuda.synchronize()
        ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts))

rows = []
for a in ['resnet50', 'densenet121', 'mobilenetv2', 'customcnn']:
    for t in ['none', 'se', 'cbam']:
        m = CXRNet(a, t, pretrained=False)          # counts/FLOPs do not depend on weight values
        c = count_params(m); f = flops(m)
        r = dict(arch=a, attention=t, Model=ARCH_LABEL[a], Attention=ATT_LABEL[t],
                 params_total_M=c['params_total'] / 1e6, params_trainable_M=c['params_trainable'] / 1e6,
                 params_attention_K=c['params_attention'] / 1e3, GFLOPs=f / 1e9, GMACs=f / 2e9,
                 cpu_latency_ms_bs1=latency_ms(m, torch.device('cpu'), 1))
        if gpu:
            r['gpu_latency_ms_bs1'] = latency_ms(m, gpu, 1)
            r['gpu_throughput_img_s_bs32'] = 32 / (latency_ms(m, gpu, 32) / 1000)
        rows.append(r); print(f"{ARCH_LABEL[a]:12s} {ATT_LABEL[t]:8s} {r['params_total_M']:.2f}M  {r['GFLOPs']:.2f} GFLOPs  CPU {r['cpu_latency_ms_bs1']:.1f} ms")
        del m
CT = pd.DataFrame(rows)

In [ ]:
# training time from notebook-02 runs (mean ± SD over seeds, per condition)
recs = []
for g in RESULT_GLOBS:
    for f in glob.glob(g, recursive=True):
        try:
            m = json.load(open(f))
            if m.get('status') == 'done' and not m.get('smoke_test') and not m.get('opts'): recs.append(m)
        except Exception: pass
if recs:
    R = pd.DataFrame(recs).drop_duplicates('run_id')
    R['train_min'] = R.total_train_sec / 60
    agg = R.groupby(['arch', 'attention', 'condition']).agg(
        n=('seed', 'nunique'), train_min_mean=('train_min', 'mean'), train_min_sd=('train_min', 'std'),
        sec_per_epoch=('mean_epoch_train_sec', 'mean'), epochs_mean=('epochs_trained', 'mean'),
        peak_gpu_mem_mb=('peak_gpu_mem_mb', 'max'), gpu=('gpu', 'first'), amp=('use_amp', 'first')).reset_index()
    for cond in agg.condition.unique():
        sub = agg[agg.condition == cond].copy()
        sub[f'train_time_{cond}'] = sub.apply(lambda r: f"{r.train_min_mean:.1f} ± {0 if pd.isna(r.train_min_sd) else r.train_min_sd:.1f} min "
                                                        f"({r.epochs_mean:.0f} ep, {r.sec_per_epoch:.0f} s/ep, n={r.n})", axis=1)
        CT = CT.merge(sub[['arch', 'attention', f'train_time_{cond}']], on=['arch', 'attention'], how='left')
    print('Hardware:', agg.gpu.unique(), '| AMP:', agg.amp.unique())
else:
    print('No notebook-02 results attached — training-time columns omitted.')

CT.to_csv(f'{OUT}/compute_table.csv', index=False)
show = CT.drop(columns=['arch', 'attention']).round(3)
display(show)
with open(f'{OUT}/compute_table.tex', 'w') as f: f.write(show.to_latex(index=False, escape=True, float_format='%.2f'))
print('CPU:', platform.processor() or platform.machine(), '| threads', torch.get_num_threads(),
      '| GPU:', torch.cuda.get_device_name(0) if gpu else 'none')